In [139]:
import os
import sys
import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_txt, read_rds, read_jsonl
from n_gram_tracing import (
    common_ngrams,
    filter_ngrams_with_outside_occurrences_in_both_texts,
    tokenize_to_tokens,
    find_all_token_ngram_spans,
    find_independent_token_ngram_spans,
    ngram_occurrence_stats
)
from utils import apply_temp_doc_id, build_metadata_df

In [150]:
# --- set your args (strings) ---
base_loc = "/Volumes/BCross"

if not os.path.exists(base_loc):
    print("Using Offline Location")
    base_loc = "/Users/user/Documents/uni_work_offline"
else:
    print("Using Volume Location")

KNOWN_LOC = f"{base_loc}/datasets/author_verification/test/Wiki/known_raw.jsonl"
UNKNOWN_LOC = f"{base_loc}/datasets/author_verification/test/Wiki/unknown_raw.jsonl"
METADATA_LOC = f"{base_loc}/datasets/author_verification/test/metadata.rds"
MODEL_LOC = f"{base_loc}/models/gpt2"

CORPUS = "Wiki"
DATA_TYPE = "test"

Using Volume Location


In [151]:
tokenizer, model = load_model(MODEL_LOC)

In [152]:
known = read_jsonl(KNOWN_LOC)
known = apply_temp_doc_id(known)
    
unknown = read_jsonl(UNKNOWN_LOC)
unknown = apply_temp_doc_id(unknown)

metadata = read_rds(METADATA_LOC)
filtered_metadata_corpus = metadata[
    (metadata['corpus'] == CORPUS)
]

In [153]:
filtered_metadata_corpus.head(5)

,problem,corpus,known_author,unknown_author
12210,Hodja_Nasreddin vs Hodja_Nasreddin,Wiki,Hodja_Nasreddin,Hodja_Nasreddin
12211,Hodja_Nasreddin vs HonestopL,Wiki,Hodja_Nasreddin,HonestopL
12212,HonestopL vs HonestopL,Wiki,HonestopL,HonestopL
12213,HonestopL vs HOOTmag,Wiki,HonestopL,HOOTmag
12214,HOOTmag vs HOOTmag,Wiki,HOOTmag,HOOTmag


In [154]:
PROBLEM = "Hodja_Nasreddin vs Hodja_Nasreddin"

In [155]:
filtered_metadata = filtered_metadata_corpus[
    filtered_metadata_corpus['problem'] ==  PROBLEM
]
agg_metadata = build_metadata_df(filtered_metadata, known, unknown)

# -----
# Get the chosen text & metadata
# -----
    
known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]

selected_known = known[known['author'] == known_author]
selected_unknown = unknown[unknown['author'] == unknown_author]

unknown_doc = selected_unknown['doc_id'].iloc[0]
unknown_text = selected_unknown['text'].iloc[0]

num_known_problems = len(selected_known)
print(f"There are {num_known_problems} known texts in the problem")
        

There are 3 known texts in the problem


In [156]:
# -------------------------------------------------------------------
# Pass 1: collect all candidate common n-grams across all known docs
# -------------------------------------------------------------------

candidate_ngram_list = []
problem_metadata_list = []

print("PASS 1: Collecting candidate common n-grams")
print("=" * 80)

for i in range(1, num_known_problems + 1):
    print(f"\nWorking on known doc {i}/{num_known_problems}")

    known_doc = selected_known["doc_id"].iloc[i - 1]
    known_text = selected_known["text"].iloc[i - 1]

    try:
        common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=True,
        )
        ngrams_found = True

    except Exception as e:
        print(f"Failed to get common n-grams for known doc {known_doc}")
        print(f"Error: {repr(e)}")

        common = []
        ngrams_found = False

    print(f"Candidate n-grams from this doc: {len(common)}")

    candidate_ngram_list.extend(common)

    problem_metadata_list.append({
        "known_doc": known_doc,
        "unknown_doc": unknown_doc,
        "ngrams_found": ngrams_found,
        "num_ngrams": len(common),
    })


print("\n" + "=" * 80)
print("PASS 1 COMPLETE")
print(f"Total candidate n-grams before dedupe: {len(candidate_ngram_list)}")

distinct_candidate_ngram_list = [
    list(x)
    for x in dict.fromkeys(tuple(x) for x in candidate_ngram_list)
]

distinct_candidate_ngram_list = sorted(
    distinct_candidate_ngram_list,
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

print(f"Distinct candidate n-grams after dedupe: {len(distinct_candidate_ngram_list)}")

print("\nFirst 10 distinct candidates:")
for g in distinct_candidate_ngram_list[:10]:
    print(g)

print("\nLast 10 distinct candidates:")
for g in distinct_candidate_ngram_list[-10:]:
    print(g)

PASS 1: Collecting candidate common n-grams

Working on known doc 1/3
Candidate n-grams from this doc: 87

Working on known doc 2/3
Candidate n-grams from this doc: 85

Working on known doc 3/3
Candidate n-grams from this doc: 76

PASS 1 COMPLETE
Total candidate n-grams before dedupe: 248
Distinct candidate n-grams after dedupe: 141

First 10 distinct candidates:
["'"]
[',']
['.']
['Ċ']
['!']
['d']
['if']
["Ġ'"]
['Ġa']
['Ġi']

Last 10 distinct candidates:
['Ġyou', 'Ġdo', 'Ġnot']
['Ġwould', 'Ġbe', 'Ġa']
['Ġdo', 'Ġnot', 'Ġhave']
['Ġthis', 'Ġis', 'Ġnot']
['Ġit', 'Ġwould', 'Ġbe']
['Ġwelcome', 'Ġto', 'Ġimprove']
['.', 'Ċ', 'this', 'Ġis']
['Ġtalk', 'Ġpage', '.', 'Ċ']
[',', 'Ġthis', 'Ġis', 'Ġnot']
[',', 'Ġand', 'Ġso', 'Ġon', '.', 'Ċ']


In [157]:
# -------------------------------------------------------------------
# Pass 2: validate full candidate list against each known doc + unknown
# -------------------------------------------------------------------

retained_ngram_list = []

print("\n" + "=" * 80)
print("PASS 2: Checking outside occurrences in BOTH known and unknown")
print("=" * 80)

for i in range(1, num_known_problems + 1):
    print(f"\nChecking known doc {i}/{num_known_problems}")

    known_doc = selected_known["doc_id"].iloc[i - 1]
    known_text = selected_known["text"].iloc[i - 1]

    try:
        valid_for_pair = filter_ngrams_with_outside_occurrences_in_both_texts(
            ngrams=distinct_candidate_ngram_list,
            known_text=known_text,
            unknown_text=unknown_text,
            tokenizer=tokenizer,
            lowercase=True,
        )

    except Exception as e:
        print(f"Failed outside-occurrence check for known doc {known_doc}")
        print(f"Error: {repr(e)}")
        valid_for_pair = []

    print(f"Valid n-grams for this known/unknown pair: {len(valid_for_pair)}")

    retained_ngram_list.extend(valid_for_pair)


print("\n" + "=" * 80)
print("PASS 2 COMPLETE")
print(f"Total retained n-grams before final dedupe: {len(retained_ngram_list)}")

distinct_ngram_list = [
    list(x)
    for x in dict.fromkeys(tuple(x) for x in retained_ngram_list)
]

distinct_ngram_list = sorted(
    distinct_ngram_list,
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

print(f"Distinct retained n-grams after final dedupe: {len(distinct_ngram_list)}")

print("\nFirst 20 retained n-grams:")
for g in distinct_ngram_list[:20]:
    print(g)

print("\nLast 20 retained n-grams:")
for g in distinct_ngram_list[-20:]:
    print(g)


PASS 2: Checking outside occurrences in BOTH known and unknown

Checking known doc 1/3
Valid n-grams for this known/unknown pair: 72

Checking known doc 2/3
Valid n-grams for this known/unknown pair: 67

Checking known doc 3/3
Valid n-grams for this known/unknown pair: 68

PASS 2 COMPLETE
Total retained n-grams before final dedupe: 207
Distinct retained n-grams after final dedupe: 119

First 20 retained n-grams:
["'"]
[',']
['Ċ']
['!']
['d']
['if']
["Ġ'"]
['Ġa']
['Ġi']
['Ġe']
['as']
['Ġas']
['Ġbe']
['Ġby']
['Ġhe']
['Ġif']
['Ġin']
['Ġis']
['Ġof']
['Ġor']

Last 20 retained n-grams:
['Ġthis', 'Ġarticle']
['Ġan', 'Ġagreement']
['Ġvery', 'Ġdifferent']
['.', 'Ċ', 'i']
['.', 'Ċ', 'he']
['.', 'Ċ', 'that']
['.', 'Ċ', 'there']
[',', 'Ġbut', 'Ġit']
[',', 'Ġyou', 'Ġare']
[',', 'Ġbut', 'Ġthis']
['Ġone', 'Ġof', 'Ġthe']
['Ġyou', 'Ġdo', 'Ġnot']
['Ġwould', 'Ġbe', 'Ġa']
['Ġdo', 'Ġnot', 'Ġhave']
['Ġit', 'Ġwould', 'Ġbe']
['Ġwelcome', 'Ġto', 'Ġimprove']
['.', 'Ċ', 'this', 'Ġis']
['Ġtalk', 'Ġpage', '.', 'Ċ

In [158]:
# -------------------------------------------------------------------
# Original angle:
# For each known doc:
#   1. get common n-grams with include_subgrams=True
#   2. immediately filter against that known text + unknown text
#   3. append valid n-grams
# Then dedupe at the end
# -------------------------------------------------------------------

single_loop_retained = []
single_loop_problem_metadata = []

print("SINGLE-LOOP METHOD: collect and filter within each known-doc loop")
print("=" * 80)

for i in range(1, num_known_problems + 1):
    print(f"\nWorking on known doc {i}/{num_known_problems}")

    known_doc = selected_known["doc_id"].iloc[i - 1]
    known_text = selected_known["text"].iloc[i - 1]

    try:
        common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=True,
        )

        print(f"Candidate n-grams from this doc: {len(common)}")

        valid_for_pair = filter_ngrams_with_outside_occurrences_in_both_texts(
            ngrams=common,
            known_text=known_text,
            unknown_text=unknown_text,
            tokenizer=tokenizer,
            lowercase=True,
        )

        print(f"Valid n-grams after within-loop filter: {len(valid_for_pair)}")

        ngrams_found = True

    except Exception as e:
        print(f"Failed for known doc {known_doc}")
        print(f"Error: {repr(e)}")

        common = []
        valid_for_pair = []
        ngrams_found = False

    single_loop_retained.extend(valid_for_pair)

    single_loop_problem_metadata.append({
        "known_doc": known_doc,
        "unknown_doc": unknown_doc,
        "ngrams_found": ngrams_found,
        "num_candidates": len(common),
        "num_valid": len(valid_for_pair),
    })


print("\n" + "=" * 80)
print("SINGLE-LOOP METHOD COMPLETE")
print(f"Total retained n-grams before dedupe: {len(single_loop_retained)}")

single_loop_distinct = [
    list(x)
    for x in dict.fromkeys(tuple(x) for x in single_loop_retained)
]

single_loop_distinct = sorted(
    single_loop_distinct,
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

print(f"Distinct retained n-grams after dedupe: {len(single_loop_distinct)}")

print("\nFirst 20 single-loop retained n-grams:")
for g in single_loop_distinct[:20]:
    print(g)

print("\nLast 20 single-loop retained n-grams:")
for g in single_loop_distinct[-20:]:
    print(g)

SINGLE-LOOP METHOD: collect and filter within each known-doc loop

Working on known doc 1/3
Candidate n-grams from this doc: 87
Valid n-grams after within-loop filter: 77

Working on known doc 2/3
Candidate n-grams from this doc: 85
Valid n-grams after within-loop filter: 74

Working on known doc 3/3
Candidate n-grams from this doc: 76
Valid n-grams after within-loop filter: 72

SINGLE-LOOP METHOD COMPLETE
Total retained n-grams before dedupe: 223
Distinct retained n-grams after dedupe: 130

First 20 single-loop retained n-grams:
["'"]
[',']
['Ċ']
['!']
['d']
['if']
["Ġ'"]
['Ġa']
['Ġi']
['Ġe']
['as']
['Ġan']
['Ġas']
['Ġbe']
['Ġby']
['Ġhe']
['Ġif']
['Ġin']
['Ġis']
['Ġof']

Last 20 single-loop retained n-grams:
['Ġan', 'Ġagreement']
['Ġvery', 'Ġdifferent']
['.', 'Ċ', 'i']
['.', 'Ċ', 'he']
['.', 'Ċ', 'that']
['.', 'Ċ', 'there']
[',', 'Ġbut', 'Ġit']
[',', 'Ġyou', 'Ġare']
[',', 'Ġthis', 'Ġis']
[',', 'Ġbut', 'Ġthis']
['Ġone', 'Ġof', 'Ġthe']
['Ġyou', 'Ġdo', 'Ġnot']
['Ġwould', 'Ġbe', 'Ġa']
['Ġ

In [159]:
# -------------------------------------------------------------------
# Compare two-pass vs single-loop
# -------------------------------------------------------------------

two_pass_set = {tuple(g) for g in distinct_ngram_list}
single_loop_set = {tuple(g) for g in single_loop_distinct}

only_in_two_pass = sorted(
    [list(g) for g in two_pass_set - single_loop_set],
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

only_in_single_loop = sorted(
    [list(g) for g in single_loop_set - two_pass_set],
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

in_both = two_pass_set & single_loop_set

print("\n" + "=" * 80)
print("COMPARISON")
print(f"Two-pass distinct n-grams     : {len(two_pass_set)}")
print(f"Single-loop distinct n-grams  : {len(single_loop_set)}")
print(f"In both                       : {len(in_both)}")
print(f"Only in two-pass              : {len(only_in_two_pass)}")
print(f"Only in single-loop           : {len(only_in_single_loop)}")

print("\nOnly in two-pass:")
for g in only_in_two_pass[:50]:
    print(g)

print("\nOnly in single-loop:")
for g in only_in_single_loop[:50]:
    print(g)


COMPARISON
Two-pass distinct n-grams     : 119
Single-loop distinct n-grams  : 130
In both                       : 119
Only in two-pass              : 0
Only in single-loop           : 11

Only in two-pass:

Only in single-loop:
['Ġon']
['the']
['Ġan']
['Ġhave']
['Ġversion']
[',', 'Ġbut']
['Ċ', 'that']
['Ġof', 'Ġthe']
['Ġdo', 'Ġnot']
['Ġis', 'Ġnot']
[',', 'Ġthis', 'Ġis']


In [160]:
def find_parent_ngrams(target_ngram, candidate_ngrams):
    """
    Find longer n-grams in candidate_ngrams that contain target_ngram.
    """
    target = tuple(target_ngram)
    parents = []

    for g in candidate_ngrams:
        g = tuple(g)

        if len(g) <= len(target):
            continue

        for i in range(len(g) - len(target) + 1):
            if g[i:i + len(target)] == target:
                parents.append(list(g))
                break

    return parents


def show_ngram_occurrences(
    text,
    target_ngram,
    candidate_ngrams,
    tokenizer,
    *,
    lowercase=True,
    window=12,
    label="text",
):
    """
    Print occurrences of target_ngram, showing whether each occurrence is
    inside a longer candidate n-gram.
    """
    tokens = tokenize_to_tokens(
        text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    target_spans = find_all_token_ngram_spans(
        tokens=tokens,
        ngram_tokens=target_ngram,
        allow_overlaps=True,
    )

    parent_ngrams = find_parent_ngrams(target_ngram, candidate_ngrams)

    parent_spans = []

    for parent in parent_ngrams:
        spans = find_all_token_ngram_spans(
            tokens=tokens,
            ngram_tokens=parent,
            allow_overlaps=True,
        )

        for span in spans:
            parent_spans.append({
                "parent_ngram": parent,
                "span": span,
            })

    print("\n" + "=" * 80)
    print(label)
    print(f"Target n-gram: {target_ngram}")
    print(f"Target occurrences: {len(target_spans)}")
    print(f"Number of possible parent n-grams: {len(parent_ngrams)}")

    if parent_ngrams:
        print("\nParent n-grams containing target:")
        for p in parent_ngrams[:20]:
            print(" ", p)

        if len(parent_ngrams) > 20:
            print(f"  ... plus {len(parent_ngrams) - 20} more")

    print("\nOccurrences:")

    for occ_i, (s, e) in enumerate(target_spans, start=1):
        left = max(0, s - window)
        right = min(len(tokens), e + window)

        context_tokens = tokens[left:right]
        context_text = tokens_to_text(context_tokens, tokenizer)

        blockers = [
            p
            for p in parent_spans
            if p["span"][0] <= s and e <= p["span"][1]
        ]

        print("\n---")
        print(f"Occurrence {occ_i}: span=({s}, {e})")
        print(f"Blocked by longer n-gram? {len(blockers) > 0}")
        print(f"Context: {context_text}")

        if blockers:
            print("Blocking parent spans:")
            for b in blockers[:10]:
                print(f"  span={b['span']} parent={b['parent_ngram']}")

            if len(blockers) > 10:
                print(f"  ... plus {len(blockers) - 10} more")

In [161]:
def compare_target_single_loop_vs_two_pass(
    target_ngram,
    selected_known,
    unknown_text,
    tokenizer,
    distinct_candidate_ngram_list,
    *,
    lowercase=True,
):
    """
    For each known doc:
      - rebuild its single-loop candidate list
      - compare stats for target under local candidates vs full two-pass candidates
    """
    rows = []

    target = tuple(target_ngram)

    print("\n" + "=" * 80)
    print(f"COMPARING TARGET: {target_ngram}")
    print("=" * 80)

    for i in range(len(selected_known)):
        known_doc = selected_known["doc_id"].iloc[i]
        known_text = selected_known["text"].iloc[i]

        common_local = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=lowercase,
        )

        target_in_local = target in {tuple(g) for g in common_local}
        target_in_full = target in {tuple(g) for g in distinct_candidate_ngram_list}

        local_known_stats = ngram_occurrence_stats(
            ngrams=common_local,
            text=known_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        ).get(target)

        local_unknown_stats = ngram_occurrence_stats(
            ngrams=common_local,
            text=unknown_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        ).get(target)

        full_known_stats = ngram_occurrence_stats(
            ngrams=distinct_candidate_ngram_list,
            text=known_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        ).get(target)

        full_unknown_stats = ngram_occurrence_stats(
            ngrams=distinct_candidate_ngram_list,
            text=unknown_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        ).get(target)

        row = {
            "known_doc": known_doc,

            "target_in_local_candidates": target_in_local,
            "target_in_full_candidates": target_in_full,

            "local_known_outside": local_known_stats["outside_count"] if local_known_stats else 0,
            "local_unknown_outside": local_unknown_stats["outside_count"] if local_unknown_stats else 0,
            "local_keep_both": (
                bool(local_known_stats and local_known_stats["keep"])
                and bool(local_unknown_stats and local_unknown_stats["keep"])
            ),

            "full_known_outside": full_known_stats["outside_count"] if full_known_stats else 0,
            "full_unknown_outside": full_unknown_stats["outside_count"] if full_unknown_stats else 0,
            "full_keep_both": (
                bool(full_known_stats and full_known_stats["keep"])
                and bool(full_unknown_stats and full_unknown_stats["keep"])
            ),

            "num_local_parent_ngrams": len(find_parent_ngrams(target_ngram, common_local)),
            "num_full_parent_ngrams": len(find_parent_ngrams(target_ngram, distinct_candidate_ngram_list)),
        }

        rows.append(row)

    out = pd.DataFrame(rows)

    print(out)

    return out

In [162]:
then_check = compare_target_single_loop_vs_two_pass(
    target_ngram=only_in_single_loop[1],
    selected_known=selected_known,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    distinct_candidate_ngram_list=distinct_candidate_ngram_list,
    lowercase=True,
)


COMPARING TARGET: ['the']
                 known_doc  target_in_local_candidates  \
0   hodja_nasreddin_text_1                       False   
1  hodja_nasreddin_text_10                        True   
2  hodja_nasreddin_text_11                       False   

   target_in_full_candidates  local_known_outside  local_unknown_outside  \
0                       True                    0                      0   
1                       True                    1                      1   
2                       True                    0                      0   

   local_keep_both  full_known_outside  full_unknown_outside  full_keep_both  \
0            False                   0                     0           False   
1             True                   1                     0           False   
2            False                   0                     0           False   

   num_local_parent_ngrams  num_full_parent_ngrams  
0                        1                       1  
1       

In [163]:
then_check[
    then_check["local_keep_both"] != then_check["full_keep_both"]
]

,known_doc,target_in_local_candidates,target_in_full_candidates,local_known_outside,local_unknown_outside,local_keep_both,full_known_outside,full_unknown_outside,full_keep_both,num_local_parent_ngrams,num_full_parent_ngrams
1,hodja_nasreddin_text_10,True,True,1,1,True,1,0,False,0,1


In [164]:
bad_doc = then_check[
    then_check["local_keep_both"] != then_check["full_keep_both"]
]["known_doc"].iloc[0]

known_text_bad = selected_known.loc[
    selected_known["doc_id"] == bad_doc,
    "text"
].iloc[0]

local_candidates_bad = common_ngrams(
    text1=known_text_bad,
    text2=unknown_text,
    tokenizer=tokenizer,
    include_subgrams=True,
    lowercase=True,
)

In [165]:
show_ngram_occurrences(
    text=known_text_bad,
    target_ngram=only_in_single_loop[1],
    candidate_ngrams=local_candidates_bad,
    tokenizer=tokenizer,
    lowercase=True,
    label=f"Known doc {bad_doc} with LOCAL candidates",
)


Known doc hodja_nasreddin_text_10 with LOCAL candidates
Target n-gram: ['the']
Target occurrences: 1
Number of possible parent n-grams: 0

Occurrences:

---
Occurrence 1: span=(0, 1)
Blocked by longer n-gram? False
Context: the soviet-german alliance before the june 21


In [141]:
show_ngram_occurrences(
    text=known_text_bad,
    target_ngram=["Ġthen"],
    candidate_ngrams=distinct_candidate_ngram_list,
    tokenizer=tokenizer,
    lowercase=True,
    label=f"Known doc {bad_doc} with FULL two-pass candidates",
)


Known doc honestopl_text_5 with FULL two-pass candidates
Target n-gram: ['Ġthen']
Target occurrences: 1
Number of possible parent n-grams: 1

Parent n-grams containing target:
  [',', 'Ġthen']

Occurrences:

---
Occurrence 1: span=(133, 134)
Blocked by longer n-gram? False
Context:  'allowed' on the 'greztky' page then and'sidney crosby's' page too.


In [168]:
show_ngram_occurrences(
    text=unknown_text,
    target_ngram=only_in_single_loop[1],
    candidate_ngrams=local_candidates_bad,
    tokenizer=tokenizer,
    lowercase=True,
    label="Unknown text with LOCAL candidates",
)

show_ngram_occurrences(
    text=unknown_text,
    target_ngram=only_in_single_loop[1],
    candidate_ngrams=distinct_candidate_ngram_list,
    tokenizer=tokenizer,
    lowercase=True,
    label="Unknown text with FULL two-pass candidates",
)


Unknown text with LOCAL candidates
Target n-gram: ['the']
Target occurrences: 1
Number of possible parent n-grams: 0

Occurrences:

---
Occurrence 1: span=(57, 58)
Blocked by longer n-gram? False
Context: 
if however you want to create 'new' article
the majority of people think they know something because they read about this

Unknown text with FULL two-pass candidates
Target n-gram: ['the']
Target occurrences: 1
Number of possible parent n-grams: 1

Parent n-grams containing target:
  ['Ċ', 'the']

Occurrences:

---
Occurrence 1: span=(57, 58)
Blocked by longer n-gram? True
Context: 
if however you want to create 'new' article
the majority of people think they know something because they read about this
Blocking parent spans:
  span=(56, 58) parent=['Ċ', 'the']
